# Assignment 2 - RAG

## Task 1 - Naive Generation

In this section, we:
1. Set up the notebook environment.
2. Load and inspect the FinanceBench dataset.
3. Filter out `metrics-generated` questions.
4. Select the first 5 `domain-relevant` and first 5 `novel-generated` questions by `financebench_id`.
5. Prepare the pipeline for naive generation using Llama-3.3-70B-Instruct via Nebius Token Factory.
6. Build the output table for manual comparison and export it to Excel.

In [ ]:
!pip -q install --upgrade pip
!pip -q install pandas openpyxl openai datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ASSIGNMENT_ROOT = Path("/content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag")
ASSIGNMENT_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = ASSIGNMENT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = ASSIGNMENT_ROOT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import json
import ast
from pathlib import Path
import time

import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from google.colab import userdata

os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
# =========================
# Task 1 Configuration
# =========================

HF_FINANCEBENCH_DATASET_NAME = "PatronusAI/financebench"
FINANCEBENCH_GITHUB_DOCS_PREFIX = "https://github.com/patronus-ai/financebench/tree/main/pdfs/"

TASK1_OUTPUT_XLSX = OUTPUT_DIR / "assignment2_naive_generation.xlsx"
FINANCEBENCH_CACHE_CSV = CACHE_DIR / "financebench_cached.csv"

# Nebius / model config
NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY", "PUT_YOUR_KEY_HERE")
NEBIUS_BASE_URL = os.getenv("NEBIUS_BASE_URL", "https://api.studio.nebius.com/v1/")
GEN_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

In [ ]:
client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)

## Task 1 assumptions

For this task, we assume:
- The FinanceBench dataset file is stored inside `assignment2_rag/data/`.
- All outputs are saved under `assignment2_rag/outputs/`.
- We use only the two required question types:
  - `domain-relevant`
  - `novel-generated`
- We exclude `metrics-generated` questions as instructed.
- We select the first 5 questions from each relevant type after sorting by `financebench_id`.

At this stage, we are only preparing the notebook structure and helper scaffolding for Task 1.

In [ ]:
def load_clean_financebench_dataset(csv_path: Path = FINANCEBENCH_CACHE_CSV) -> pd.DataFrame:
    try:
        df = pd.read_csv(csv_path)
        print(f"Loaded dataset from local cache: {csv_path}")
        return df

    except FileNotFoundError:
        print("Local CSV not found. Loading dataset from Hugging Face...")

        ds = load_dataset(HF_FINANCEBENCH_DATASET_NAME)
        df = ds["train"].to_pandas()

        df = df[df["question_type"] != "metrics-generated"].copy()

        print(f"Loaded {len(df)} rows from Hugging Face after removing metrics-generated questions.")

        mask = df["doc_name"].notna() & (df["doc_name"].astype(str).str.strip() != "")
        df.loc[mask, "doc_link"] = FINANCEBENCH_GITHUB_DOCS_PREFIX + df.loc[mask, "doc_name"].astype(str) + ".pdf"

        df.to_csv(csv_path, index=False)
        print(f"Cached cleaned dataset to: {csv_path}")

        return df


def inspect_dataset(df: pd.DataFrame) -> None:
    print("Shape:")
    print(df.shape)
    print()

    print("Columns:")
    print(df.columns.tolist())
    print()

    print("Question type counts:")
    print(df["question_type"].value_counts(dropna=False))
    print()

    print("Unique question types:")
    print(df["question_type"].unique())
    print()

    print("Sample rows:")
    print(df.head(5))
    print()


def select_questions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("financebench_id").copy()

    df_domain_relevant = df[df["question_type"] == "domain-relevant"].head(5)
    df_novel_generated = df[df["question_type"] == "novel-generated"].head(5)

    if len(df_domain_relevant) < 5 or len(df_novel_generated) < 5:
        raise ValueError("Not enough questions.")

    return pd.concat([df_domain_relevant, df_novel_generated], ignore_index=True)


def build_naive_prompt(question: str) -> list:
    return [
        {"role": "system", "content": "Answer the user's question as best as you can."},
        {"role": "user", "content": question}
    ]

In [ ]:
def generate_naive_answer(question: str, model: str = GEN_MODEL) -> str:
    question_prompt = build_naive_prompt(question)

    response = client.chat.completions.create(
        model=model,
        messages=question_prompt,
        temperature=0.0,
    )

    return response.choices[0].message.content

In [ ]:
def build_results_table(questions_df: pd.DataFrame) -> pd.DataFrame:
    retults = []
    for row in questions_df.itertuples():
      question = row.question
      question_type = row.question_type
      financebench_id = row.financebench_id
      ground_truth = row.answer
      naive_answer = generate_naive_answer(question)
      retults.append({
          "financebench_id": financebench_id,
          "question_type": question_type,
          "question": question,
          "naive_answer": naive_answer,
          "ground_truth": ground_truth,
          "verdict": None,
      })

    return pd.DataFrame(retults)


def save_results_to_excel(results_df: pd.DataFrame, output_path: Path, always_save: bool = False) -> None:
    if not output_path.exists() or always_save:
        results_df.to_excel(output_path, index=False)
        print(f"Saved results to Excel: {output_path}")

In [ ]:
# =========================
# Task 1 Execution Flow
# =========================

# Step 1: Load & Clean dataset
financebench_df = load_clean_financebench_dataset()

# Step 2: Inspect dataset
inspect_dataset(financebench_df)

# Step 3: Select first 5 domain-relevant and first 5 novel-generated by financebench_id
questions_df = select_questions(financebench_df)

# Step 4: Build theresults table with naive model outputs
results_df = build_results_table(questions_df)

# Step 5: Save to Excel
save_results_to_excel(results_df, TASK1_OUTPUT_XLSX)

# Step 6: Display the final table
display(results_df.head(10))

Loaded dataset from local cache: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/cache/financebench_cached.csv
Shape:
(100, 15)

Columns:
['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']

Question type counts:
question_type
domain-relevant    50
novel-generated    50
Name: count, dtype: int64

Unique question types:
['domain-relevant' 'novel-generated']

Sample rows:
         financebench_id company       doc_name    question_type  \
0  financebench_id_00499      3M    3M_2022_10K  domain-relevant   
1  financebench_id_01226      3M    3M_2022_10K  domain-relevant   
2  financebench_id_01865      3M    3M_2022_10K  novel-generated   
3  financebench_id_00807      3M  3M_2023Q2_10Q  domain-relevant   
4  financebench_id_00941      3M  3M_2023Q2_10Q  domain-relevant   

          

,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,"Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","To determine if Corning has positive working capital based on FY2022 data, we need to calculate the working capital. Working capital is calculated as current assets minus current liabilities.\n\nAccording to Corning's FY2022 annual report (10-K filing), the company's current assets and current liabilities are:\n\n* Current assets: $8,144 million (as of December 31, 2022)\n* Current liabilities: $4,341 million (as of December 31, 2022)\n\nWorking capital = Current assets - Current liabilities\n= $8,144 million - $4,341 million\n= $3,803 million\n\nSince the result is positive, Corning has positive working capital based on FY2022 data.\n\nWorking capital is a relevant metric for Corning, as it is a manufacturing company with significant inventories, accounts receivable, and accounts payable. The company's ability to manage its working capital is important to ensure it has sufficient liquidity to meet its short-term obligations and invest in its operations. However, it's worth noting that Corning's working capital requirements may be influenced by its business model, industry, and supply chain, so it's essential to consider these factors when analyzing the company's working capital.",Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers only operating current assets and current liabilities that were clearly shown in the balance sheet.,None
1,financebench_id_00070,domain-relevant,"Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","To determine if American Water Works has positive working capital, we need to look at its FY2022 financial data. Working capital is calculated as current assets minus current liabilities.\n\nAccording to American Water Works' FY2022 annual report (10-K filing), the company's current assets and current liabilities are:\n\n* Current assets: $1.43 billion\n* Current liabilities: $2.33 billion\n\nUsing these values, we can calculate working capital as follows:\n\nWorking capital = Current assets - Current liabilities\n= $1.43 billion - $2.33 billion\n= -$0.90 billion\n\nSince the result is negative, American Water Works does not have positive working capital based on FY2022 data.\n\nHowever, it's worth noting that working capital might not be the most relevant metric for a utility company like American Water Works. Utility companies often have significant capital expenditures and long-term debt, which can affect their current liabilities and make working capital a less useful indicator of their financial health.\n\nIn the case of American Water Works, the company's business model involves investing in long-term assets like water treatment plants and distribution infrastructure, which can result in significant long-term debt and capital lease obligations. As a result, the company's current liabilities may not be a good indicator of its ability to meet its short-term obligations.\n\nMore relevant metrics for evaluating American Water Works' financial health might include its debt-to-equity ratio, interest coverage ratio, and cash flow from operations. These metrics can provide a better understanding of the company's ability to manage its debt, generate cash, and invest in its business.","No, American Water Works had negative working capital of -$1561M in FY 2022.",None
2,financebench_id_00080,domain-relevant,"Does Paypal have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","To determine if PayPal has positive working capi

## Task 1 discussion

1. The naive model explicitly refused or asked for more information in cases where the question depended on company-specific filing context that was not provided, such as the cash and cash equivalents question, where it said it needed the company name or source data. This is expected because the baseline prompt included only the question and no retrieved evidence.

2. When the model answered confidently, it was often only partially correct rather than fully correct. It frequently identified the right general direction, such as whether working capital was positive or negative, but used the wrong magnitude or unsupported reasoning. In a few cases it hallucinated specific figures or misread the metric entirely, such as the Pfizer spin-off and PPNE questions.

3. There is a noticeable pattern by question type. The `domain-relevant` questions performed better overall, with some correct and several partially correct answers. The `novel-generated` questions failed more often, with more wrong or refused answers. A likely reason is that the novel-generated questions depend more heavily on exact document-specific evidence, which the model could not access in the naive setting.

## Task 2 – RAG pipeline components

**Indexing** converts the source documents into a searchable knowledge base by splitting them into chunks, embedding them, and storing them in a vector store. It can fail if chunking breaks important context, parses data purely, is missing metadata, or if embeddings do not capture the meaning well. For example, a relevant table may end up separated from the main text. This stage usually happens once offline.

**Retrieval** takes a user query and finds the most relevant chunks from the indexed collection. It can fail if the query wording does not match the wording in the filing, if the wrong document is retrieved, or if the correct evidence is ranked below top-k. This happens for every query.

**Generation** uses the retrieved chunks plus the query to produce the final answer. It can fail if the model ignores the context, mixes facts from different chunks, or hallucinates when the retrieved evidence is incomplete. For example, giving a specific number that does not appear in the context. This also happens for every query.

# Task 3 – Document Embeddings

In this section, we:
1. Install the libraries needed for PDF loading, chunking, embeddings, and vector search.
2. Download the relevant source PDF files.
3. Load each PDF page by page and attach standardized metadata.
4. Split the page-level documents into chunks.
5. Embed the document chunks using the selected embedding model.
6. Build a FAISS vector store for efficient similarity-based retrieval.
7. Save the vector store to disk for reuse.
8. Run a few dataset questions as queries to inspect the retrieved chunks and assess retrieval quality.

In [ ]:
!pip -q install requests pypdf faiss-cpu sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import re
import torch
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
# =========================
# Task 3 Configuration
# =========================

PDF_DIR = ASSIGNMENT_ROOT / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

INDEX_DIR = ASSIGNMENT_ROOT / "indices"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CACHED_PDFS_CSV = CACHE_DIR / "cached_pdfs.csv"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

FAISS_INDEX_NAME = "faiss_chunk1000"
FAISS_INDEX_PATH = INDEX_DIR / FAISS_INDEX_NAME

In [ ]:
def to_raw_github_url(url: str) -> str:
    return url.replace("https://github.com/", "https://raw.githubusercontent.com/").replace("/blob/", "/").replace("/tree/", "/")


def download_relevant_pdfs(df: pd.DataFrame) -> pd.DataFrame:
    if CACHED_PDFS_CSV.exists():
        return pd.read_csv(CACHED_PDFS_CSV)

    pdf_rows = []

    unique_df = df.drop_duplicates(subset=["doc_name"]).copy()

    for row in unique_df.itertuples(index=False):
        doc_name = row.doc_name
        doc_link = row.doc_link
        company = row.company
        doc_period = row.doc_period

        local_pdf_path = PDF_DIR / f"{doc_name}.pdf"

        raw_url = to_raw_github_url(doc_link)
        print(f"Downloading {raw_url} -> {local_pdf_path}")

        response = requests.get(to_raw_github_url(doc_link), timeout=60)
        response.raise_for_status()

        with open(local_pdf_path, "wb") as f:
            f.write(response.content)

        pdf_rows.append({
            "doc_name": doc_name,
            "company": company,
            "doc_period": doc_period,
            "local_pdf_path": str(local_pdf_path),
        })

    df = pd.DataFrame(pdf_rows)
    df.to_csv(CACHED_PDFS_CSV, index=False)
    print(f"Cached downloaded PDFs to: {CACHED_PDFS_CSV}")
    return df



def load_pdf_pages_with_metadata(pdf_path: str, doc_name: str, company: str, doc_period: str):
    docs = PyPDFLoader(pdf_path, mode="page").load()
    print(f"Loaded {len(docs)} pages from {pdf_path}")

    for page in docs:
      original_page_number = page.metadata.get("page", 0)
      page.metadata = {
          "doc_name": doc_name,
          "company": company,
          "doc_period": doc_period,
          "page_number": int(original_page_number),
      }

    return docs



def load_all_pdf_pages(pdf_df: pd.DataFrame):
    all_pdfs = []

    for row in pdf_df.itertuples(index=False):
        docs = load_pdf_pages_with_metadata(
            row.local_pdf_path,
            row.doc_name,
            row.company,
            row.doc_period,
        )

        all_pdfs.extend(docs)

    return all_pdfs


def inspect_page_documents(page_docs, n: int = 3) -> None:
    print(f"Total number of page documents: {len(page_docs)}")

    for doc in page_docs[:n]:
        print(f"Metadata: {doc.metadata}")


In [ ]:
def build_text_splitter(chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def chunk_page_documents(page_docs, text_splitter):
    return text_splitter.split_documents(page_docs)


def inspect_chunk_documents(chunk_docs, n: int = 5) -> None:
    print("Total chunk documents:", len(chunk_docs))
    print()

    for i, chunk_doc in enumerate(chunk_docs[:n]):
        print(f"Chunk #{i}")
        print("Metadata:")
        print(chunk_doc.metadata)
        print()
        print("Content preview:")
        print(chunk_doc.page_content[:500])
        print("\n" + "=" * 80 + "\n")

In [ ]:
def build_embedding_model(model_name: str = EMBED_MODEL_NAME):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={
            "batch_size": 32,
            "normalize_embeddings": True
        }
    )


def build_faiss_vectorstore(chunk_docs, embedding_model, index_file_path: Path = FAISS_INDEX_PATH):
    if index_file_path.exists():
        print(f"Loading existing FAISS index from: {index_file_path}")
        return load_faiss_vectorstore(index_file_path, embedding_model)

    print(f"Building FAISS index {index_file_path}...")
    return FAISS.from_documents(chunk_docs, embedding_model)

def save_faiss_vectorstore(vectorstore, save_path: Path) -> None:
    vectorstore.save_local(save_path)
    print(f"Saved FAISS index to: {save_path}")

def load_faiss_vectorstore(save_path: Path, embedding_model):
    folder_path = str(save_path)
    vectorstore = FAISS.load_local(
        folder_path=folder_path,
        embeddings=embedding_model,
        allow_dangerous_deserialization=True,
    )
    print(f"FAISS vector store loaded from: {folder_path}")
    return vectorstore

In [ ]:
# =========================
# Task 3 Execution Flow - Part A
# Download all relevant PDFs
# =========================

# Step 1: Load the cleaned filtered dataset
filtered_df = load_clean_financebench_dataset()

# Step 2: Download all relevant PDFs using doc_link
pdf_df = download_relevant_pdfs(filtered_df)

# Step 3: Display the downloaded PDF table
print(pdf_df.head(10).to_string())

Loaded dataset from local cache: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/cache/financebench_cached.csv
                         doc_name           company  doc_period                                                                                                      local_pdf_path
0                     3M_2022_10K                3M        2022                     /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/3M_2022_10K.pdf
1                   3M_2023Q2_10Q                3M        2023                   /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/3M_2023Q2_10Q.pdf
2                  ADOBE_2022_10K             Adobe        2022                  /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/ADOBE_2022_10K.pdf
3                    AES_2022_10K   AES Corporation        2022                    /content/drive/MyDrive/nebius academy/assignments/guy-peer_

In [ ]:
# =========================
# Task 3 Execution Flow - Part B
# Load PDFs page by page with metadata
# =========================

# Step 4: Load all PDFs as page-level Documents
page_docs = load_all_pdf_pages(pdf_df)

# Step 5: Inspect loaded page documents
inspect_page_documents(page_docs, n=3)

Loaded 252 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/3M_2022_10K.pdf
Loaded 92 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/3M_2023Q2_10Q.pdf
Loaded 99 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/ADOBE_2022_10K.pdf
Loaded 257 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/AES_2022_10K.pdf
Loaded 9 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/AMCOR_2022_8K_dated-2022-07-01.pdf
Loaded 156 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/AMCOR_2023_10K.pdf
Loaded 57 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/AMCOR_2023Q2_10Q.pdf
Loaded 14 pages from /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/pdfs/AMCOR_2023Q4_EARNINGS.pdf
Loaded 121 pages fr

In [ ]:
# =========================
# Task 3 Execution Flow - Part C
# Chunk page-level documents
# =========================

# Step 6: Build the text splitter
text_splitter = build_text_splitter()

# Step 7: Split page-level documents into chunk-level documents
chunk_docs = chunk_page_documents(page_docs, text_splitter)

# Step 8: Inspect chunked documents
inspect_chunk_documents(chunk_docs, n=5)

Total chunk documents: 24218

Chunk #0
Metadata:
{'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}

Content preview:
T able of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 
10-K
 
 
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
 
December 31
, 2022
or
o
 
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from __________ to __________
Commission file number
 
1-3285
3M COMPANY
State of Incorporation:
 
Delaware
I.R.S. Employer Identification N


Chunk #1
Metadata:
{'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}

Content preview:
0.950% Notes due 2023
MMM23
New York Stock Exchange
1.500% Notes due 2026
MMM26
New York Stock Exchange
1.750% Notes due 2030
MMM30
New York Stock Exchange
1.500% Notes due 2031
MMM31
New York Stock Exchange
Note: The common st

In [ ]:
# =========================
# Task 3 Execution Flow - Part D
# Embed chunks and build FAISS vector store
# =========================

# Step 9: Build the embedding model
embedding_model = build_embedding_model()
print("Embedding model loaded. Checking it: \n")
sample_vec = embedding_model.embed_query("What was Corning's working capital in 2022?")
print("type: ", type(sample_vec))
print("vec size: ", len(sample_vec))
print("sub vec: ", sample_vec[:10])
print()

# Step 10: Build the FAISS vector store
print("Number of page docs:", len(page_docs))
print("Number of chunk docs:", len(chunk_docs))
print("Building FAISS vectorstore...")
vectorstore = build_faiss_vectorstore(chunk_docs, embedding_model)
print("FAISS vectorstore built.")

# Step 11: Save the FAISS index locally
save_faiss_vectorstore(vectorstore, FAISS_INDEX_PATH)

# Step 12: Reload the saved vector store
loaded_vectorstore = load_faiss_vectorstore(FAISS_INDEX_PATH, embedding_model)

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Checking it: 

type:  <class 'list'>
vec size:  384
sub vec:  [-0.013459251262247562, -0.0282730795443058, 0.027585197240114212, -0.0005004752893000841, 0.05635704845190048, -0.003603394376114011, 0.03406394273042679, -0.03735504299402237, -0.0476704016327858, -0.028891632333397865]

Number of page docs: 5448
Number of chunk docs: 24218
Building FAISS vectorstore...
Loading existing FAISS index from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000
FAISS vectorstore built.
Saved FAISS index to: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000


In [ ]:
### Sanity check
results = loaded_vectorstore.similarity_search("What was Corning working capital in 2022?", k=3)

for i, doc in enumerate(results):
    print(f"Result #{i}")
    print(doc.metadata)
    print(doc.page_content[:400])
    print("\n" + "=" * 80 + "\n")

Result #0
{'doc_name': 'CORNING_2022_10K', 'company': 'Corning', 'doc_period': 2022, 'page_number': 70}
Corning periodically assesses the operating efficiency and cost structure of the Company’s asset base and global workforce and takes appropriate actions to aligncorporate resources with the business environment. 2022 Corning recorded $414 million in severance, accelerated depreciation, asset write-offs and other related charges for the year ended December 31, 2022. Capacityoptimization charges inc


Result #1
{'doc_name': 'CORNING_2022_10K', 'company': 'Corning', 'doc_period': 2022, 'page_number': 60}
Table of Contents Consolidated Statements of Cash Flows Corning Incorporated and Subsidiary Companies
 
  Year ended December 31,  
(in millions)  2022   2021   2020


Result #2
{'doc_name': 'CORNING_2022_10K', 'company': 'Corning', 'doc_period': 2022, 'page_number': 57}
Table of Contents  Consolidated Statements of Income Corning Incorporated and Subsidiary Companies
  
  Year ended De

In [ ]:
def extract_evidence_page_nums(evidence_str: str) -> list[str]:
    return re.findall(r"'evidence_page_num'\s*:\s*(\d+)", evidence_str)

def retrieve_top_k_chunks(vectorstore, query: str, k: int = 3):
    return vectorstore.similarity_search(query, k=k)

def inspect_retrieving_results(question_row: pd.Series, retrieved_docs, with_content: bool = False):
    print("Question: ", question_row["question"])
    print()

    print("Expected doc_name: ", question_row["doc_name"])
    print()

    print("Expected evidence page nums: ", ", ".join(extract_evidence_page_nums(question_row["evidence"])))
    print()

    for i, doc in enumerate(retrieved_docs):
        print(f"Retrieved chunk #{i+1}")
        print("Metadata:")
        print(doc.metadata)
        print()
        if with_content:
          print("Content preview:")
          print(doc.page_content[:1500])
        print("\n" + "=" * 100 + "\n")


In [ ]:
financebench_df = load_clean_financebench_dataset()
sample_questions_df = financebench_df.head(5).copy()
for _, row in sample_questions_df.iterrows():
  retrieved_docs = retrieve_top_k_chunks(
      vectorstore=vectorstore,
      query=row["question"],
      k=5,
  )
  inspect_retrieving_results(row, retrieved_docs, True)

Loaded dataset from local cache: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/cache/financebench_cached.csv
Question:  Is 3M a capital-intensive business based on FY2022 data?

Expected doc_name:  3M_2022_10K

Expected evidence page nums:  47, 49, 51

Retrieved chunk #1
Metadata:
{'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 37}

Content preview:
$
(548)
Refer to the preceding “Total Debt” and “Cash, Cash Equivalents and Marketable Securities” sections for additional details.
Balance Sheet:
3M’s strong balance sheet and liquidity provide the Company with significant flexibility to fund its numerous opportunities going forward. The Company will continue to invest
in its operations to drive growth, including continual review of acquisition opportunities.
The Company uses working capital measures that place emphasis and focus on certain working capital assets, such as accounts receivable and inventory activity.
Working capita

## Task 3 observations

Retrieval was only partially successful. Some top-k chunks came from the correct document, but several results were pulled from the wrong filing (`3M_2023Q2_10Q` instead of `3M_2022_10K`). The retrieved chunks often matched the general topic, but they did not consistently overlap with the dataset evidence text or the annotated `evidence_page_num`.

This suggests that the current retriever captures coarse semantic similarity, but not the exact supporting evidence well enough. As a result, downstream RAG failures may be caused by retrieval misses rather than generation alone.

# Task 4 – Build a RAG Pipeline

In this section, we build a simple Retrieval-Augmented Generation (RAG) pipeline.  
Given a user query, the pipeline first retrieves the most relevant chunks from the FAISS vector store built in Task 3, and then passes both the query and the retrieved context to the generation model to produce a final answer.

We use the same embedding model as in Task 3 (`BAAI/bge-small-en-v1.5`) to embed the query, and the same generation model as in Task 1 (`Llama-3.3-70B-Instruct`) to generate the response.  
To make the setup reliable, we format the retrieved chunks clearly in the prompt, include each chunk’s `doc_name` and `page_number`, and instruct the model to answer only from the provided context, stay concise, cite the source document, and explicitly say when the context is insufficient.

In [ ]:
RAG_SYSTEM_PROMPT = """You are a financial QA assistant working only from the provided context.

Instructions:
1. Answer only using the provided context.
2. If the context does not contain enough information to answer, say so explicitly and do not guess.
3. Keep the answer concise.
4. Cite the source document for each factual claim using the document name in parentheses.
5. If multiple chunks are relevant, combine them carefully and do not invent missing details.
"""

In [ ]:
def format_retrieved_chunks(retrieved_docs) -> str:
    if not retrieved_docs:
        return "No relevant context was retrieved."

    formatted_chunks = []

    for i, doc in enumerate(retrieved_docs, start=1):
        doc_name = doc.metadata.get("doc_name", "UNKNOWN_DOC")
        page_number = doc.metadata.get("page_number", "UNKNOWN_PAGE")
        chunk_text = doc.page_content.strip()

        formatted_chunk = f"""
          ### Chunk {i}
          Document: {doc_name}
          Page: {page_number}
          Content: {chunk_text}
        """.strip()

        formatted_chunks.append(formatted_chunk)

    return "\n\n---\n\n".join(formatted_chunks)

In [ ]:
def build_rag_user_prompt(query: str, retrieved_docs) -> str:
    context_block = format_retrieved_chunks(retrieved_docs)

    prompt = f"""
        User question:
        {query}

        Retrieved context:
        {context_block}

        Answer the question using only the retrieved context above.
    """.strip()

    return prompt

In [ ]:
embedding_model = build_embedding_model()
vectorstore = load_faiss_vectorstore(FAISS_INDEX_PATH, embedding_model)

def build_rag_messages(query: str, retrieved_docs) -> list[dict]:
    return [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": build_rag_user_prompt(query, retrieved_docs)},
    ]

def answer_with_rag(query: str, k: int = 4, print_user_prompt: bool = False) -> dict:
    retrieved_docs = vectorstore.similarity_search(query, k=k)
    rag_messages = build_rag_messages(query, retrieved_docs)

    if print_user_prompt:
      print("User prompt: ", rag_messages[1]["content"])

    response = client.chat.completions.create(
        model=GEN_MODEL,
        temperature=0.0,
        messages=build_rag_messages(query, retrieved_docs),
    )

    answer = response.choices[0].message.content.strip()

    retrieved_chunks = [
        {
          "doc_name": doc.metadata.get("doc_name"),
          "page_number": doc.metadata.get("page_number"),
          "content": doc.page_content.strip(),
        }
        for doc in retrieved_docs
    ]

    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
    }


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000


In [ ]:
## Sanity check
financebench_df = load_clean_financebench_dataset()

query = financebench_df.loc[5]["question"]
print("QUERY:")
print(query)
print()

result = answer_with_rag(query, k=4)
print("ANSWER:")
print(result["answer"])
print()

print("RETRIEVED CHUNKS:")
for chunk in result["retrieved_chunks"]:
    print(chunk["doc_name"], chunk["page_number"])

Loaded dataset from local cache: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/cache/financebench_cached.csv
QUERY:
Does 3M maintain a stable trend of dividend distribution?

ANSWER:
Yes, 3M maintains a stable trend of dividend distribution, having paid dividends since 1916 and marking the 65th consecutive year of dividend increases in 2023 (3M_2023Q2_10Q, 3M_2022_10K).

RETRIEVED CHUNKS:
3M_2023Q2_10Q 72
3M_2022_10K 40
3M_2023Q2_10Q 69
3M_2022_10K 36


# Task 5 - Run & Compare

In [ ]:
# =========================
# Task 5 Configuration
# =========================

TASK5_OUTPUT_XLSX = OUTPUT_DIR / "assignment2_run_and_compare.xlsx"

In [ ]:
def read_naive_generation() -> pd.DataFrame:
    return pd.read_excel(TASK1_OUTPUT_XLSX)

def generate_rag_answers_df(naive_generation_df: pd.DataFrame, k: int = 4) -> pd.DataFrame:
    results = []

    try:
      for row in naive_generation_df.itertuples(index=False):
          result = answer_with_rag(row.question, k=k)

          retrieved_sources = [
              {
                  "doc_name": chunk["doc_name"],
                  "page_number": chunk["page_number"],
                  "content": chunk["content"],
              }
              for chunk in result["retrieved_chunks"]
          ]

          results.append({
              "financebench_id": row.financebench_id,
              "question_type": row.question_type,
              "question": row.question,
              "naive_answer": row.naive_answer,
              "RAG_answer": result["answer"],
              "ground_truth": row.ground_truth,
              "retrieved_sources": str(retrieved_sources),
          })
    except Exception as e:
            results.append({
                "financebench_id": row.financebench_id,
                "question_type": row.question_type,
                "question": row.question,
                "naive_answer": row.naive_answer,
                "rag_answer": f"ERROR: {e}",
                "ground_truth": row.ground_truth,
                "retrieved_sources": "",
            })
    return pd.DataFrame(results)

def save_rag_answers_df(df: pd.DataFrame) -> None:
    df.to_excel(TASK5_OUTPUT_XLSX, index=False)
    print(f"Saved RAG answers to: {TASK5_OUTPUT_XLSX}")


In [ ]:
# =======================
# Task 5 Execution Flow
# =======================
if not TASK5_OUTPUT_XLSX.exists():
  naive_generation_df = read_naive_generation()
  rag_answers_df = generate_rag_answers_df(naive_generation_df)
  save_rag_answers_df(rag_answers_df)
  rag_answers_df.head()

## Task 5 Discussion

**Did RAG help?**  
Yes, mainly in cases where the naive model refused or hallucinated and retrieval surfaced the relevant filing. For example, in the Best Buy cash-and-cash-equivalents question, the naive model refused due to missing company context, while RAG retrieved the correct FY2023 and Q2 FY2024 filings and produced a grounded answer. RAG also improved the MGM EBITDA question by retrieving the correct MGM 2022 filing and giving a source-based answer.

**Did RAG hurt?**  
Yes, in several cases retrieval brought the wrong filing or irrelevant chunks, which made the final answer worse than the naive baseline. For example, Corning, American Water Works, and PayPal working-capital questions all suffered because RAG retrieved incomplete or misleading chunks and answered that the context was insufficient, even though the naive model at least captured the correct direction. JPM gross margin also got worse because RAG relied on irrelevant retrieved chunks instead of using the simpler reasoning that gross margin is not a useful metric for a bank.

**Patterns by question type**  
RAG helped more on `novel-generated` questions when the answer depended on exact document evidence, since retrieval could supply the missing context. However, for `domain-relevant` questions, the naive model was sometimes already directionally correct from general financial knowledge, and RAG could hurt when retrieval missed the right page or mixed in the wrong filing. A likely reason is that document-specific questions benefit more from retrieval, while broad finance questions are more vulnerable to retrieval noise.

# Task 6 – End-to-End Automated Evaluation

In this section, we run all FinanceBench questions through the RAG pipeline and evaluate each answer automatically with three measures:

1. **Correctness** – scored by an LLM judge against the ground truth answer.
2. **Faithfulness** – scored with RAGAS on a subset of 20 examples due to token cost.
3. **Page hit@k** – computed directly from retrieval metadata by checking whether at least one retrieved chunk at k ∈ {1, 3, 5} came from the expected evidence page.

The final output includes both a per-question evaluation spreadsheet and aggregate summary metrics.

In [ ]:
!pip -q install ragas

In [ ]:
import pandas as pd
from typing import Any
from ragas import SingleTurnSample
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2304/85026616.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


In [ ]:
JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"
RAG_CACHED_CSV = CACHE_DIR / "rag_cached.csv"
TASK6_OUTPUT_XLSX = OUTPUT_DIR / "assignment2_evaluation.xlsx"

In [ ]:
def parse_value(value: Any) -> list[dict]:
    if isinstance(value, list):
        return value

    if value is None:
        return []

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []

        try:
            return json.loads(value)
        except Exception:
            try:
                return ast.literal_eval(value)
            except Exception:
                return []

    return []


def extract_evidence_page_nums_from_evidence(evidence_value: Any) -> list[int]:
    evidence_items = parse_value(evidence_value)

    page_nums = []
    for item in evidence_items:
        if isinstance(item, dict) and "evidence_page_num" in item:
            try:
                page_nums.append(int(item["evidence_page_num"]))
            except Exception:
                pass

    return page_nums

In [ ]:
def build_rag_df(financebench_df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    if RAG_CACHED_CSV.exists():
        df = pd.read_csv(RAG_CACHED_CSV)
        df["retrieved_sources"] = df["retrieved_sources"].apply(parse_value)
        df["expected_page_nums"] = df["expected_page_nums"].apply(parse_value)
        print(f"Loaded RAG results from cache: {RAG_CACHED_CSV}")
        return df

    results = []

    print(f"Running RAG on {len(financebench_df)} questions with k={k}...")
    for row in financebench_df.itertuples(index=False):
        result = answer_with_rag(row.question, k=k)

        expected_page_nums = extract_evidence_page_nums_from_evidence(row.evidence)

        results.append({
            "financebench_id": row.financebench_id,
            "question": row.question,
            "ground_truth": row.answer,
            "evidence": row.evidence,
            "expected_page_nums": expected_page_nums,
            "rag_answer": result["answer"],
            "retrieved_sources": result["retrieved_chunks"],
        })
        print(f"Completed question '{row.question}'")

    df = pd.DataFrame(results)
    df.to_csv(RAG_CACHED_CSV, index=False)
    print(f"Saved RAG results to: {RAG_CACHED_CSV}")
    return df

In [ ]:
def has_page_hit_at_k(expected_pages: list[int], retrieved_sources: list[dict], k: int) -> int:
    expected_pages_set = set(expected_pages)

    top_k_sources = retrieved_sources[:k]
    for chunk in top_k_sources:
        page_number = chunk.get("page_number")
        if page_number in expected_pages_set:
            return 1

    return 0

In [ ]:
def has_page_hit_at_k_for_row(row: pd.Series, k: int = 3) -> pd.Series:
    return has_page_hit_at_k(row["expected_page_nums"], row["retrieved_sources"], k=k)

def add_page_hit_at_k_columns(rag_df: pd.DataFrame) -> pd.DataFrame:
    if RAG_CACHED_CSV.exists() and {"page_hit_at_1", "page_hit_at_3", "page_hit_at_5"}.issubset(rag_df.columns):
        print("Pag hit at {k} columns exist. skipping...")
        return rag_df

    df = rag_df.copy()
    for k in (1, 3, 5):
        df[f"page_hit_at_{k}"] = df.apply(has_page_hit_at_k_for_row, axis=1, args=(k,))

    return df

In [ ]:
CORRECTNESS_JUDGE_SYSTEM_PROMPT = """
  You are an evaluation judge.

  Your task is to compare a model answer against the ground truth answer for a financial question.

  Return only a single numeric score:
  - 1.0 if the answer is correct
  - 0.5 if the answer is partially correct
  - 0.0 if the answer is incorrect

  Guidelines:
  - Give 1.0 if the model answer matches the essential substance of the ground truth.
  - Give 0.5 if it gets the main direction right but misses important details or includes minor inaccuracies.
  - Give 0.0 if it is wrong, unsupported, or fails to answer the question.
  Return only the number and nothing else.
""".strip()

In [ ]:
def build_correctness_judge_prompt(question: str, ground_truth: str, rag_answer: str) -> str:
    prompt = f"""
      Question:
      {question}

      Ground truth answer:
      {ground_truth}

      Model answer:
      {rag_answer}

      Evaluate the model answer against the ground truth and return only one number:
      - 1.0
      - 0.5
      - 0.0
      """.strip()

    return prompt

In [ ]:
def judge_correctness(question: str, ground_truth: str, rag_answer: str) -> str:
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": CORRECTNESS_JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": build_correctness_judge_prompt(question, ground_truth, rag_answer)},
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    print("judge_correctness: ", question, "(", content, ")")

    try:
        time.sleep(5)
        return float(content)
    except Exception:
        print(f"Failed to parse response: {content}")
        return 0.0

In [ ]:
def add_correctness_scores(rag_df: pd.DataFrame) -> pd.DataFrame:
    if RAG_CACHED_CSV.exists() and "correctness" in rag_df.columns:
        print("Correctness column exist. skipping...")
        return rag_df

    df = rag_df.copy()

    correctness_scores = []

    print("Evaluating correctness...")
    for row in df.itertuples(index=False):
        score = judge_correctness(
            question=row.question,
            ground_truth=row.ground_truth,
            rag_answer=row.rag_answer,
        )
        correctness_scores.append(score)

    df["correctness"] = correctness_scores
    print("Done evaluating correctness.")

    return df

In [ ]:
## Limit context otherwise ragas breaks down
def retrieved_sources_to_contexts(retrieved_sources: list[dict], max_context_chars: int = 5000) -> list[str]:
    contexts = []

    for chunk in retrieved_sources:
        text = chunk.get("content", "")
        if text:
            contexts.append(text[:max_context_chars])

    return contexts

In [ ]:
def add_faithfulness_scores(rag_df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    if RAG_CACHED_CSV.exists() and "faithfulness" in rag_df.columns:
        print("Faithfulness column already exists. Skipping...")
        return rag_df

    df = rag_df.copy()
    df["faithfulness"] = pd.NA

    scorer = Faithfulness(
        llm=llm_factory(
            JUDGE_MODEL,
            client=client,
            max_tokens=5000,
        )
    )

    for i in range(min(n, len(df))):
        row = df.iloc[i]

        question = row["question"]
        answer = row["rag_answer"]

        sample = SingleTurnSample(
            user_input=question,
            response=answer,
            retrieved_contexts=retrieved_sources_to_contexts(row["retrieved_sources"]),
        )

        score = scorer.single_turn_score(sample)
        print(f"Faithfulness score {score} for question '{question}', RAG answer '{answer}'")
        df.loc[df.index[i], "faithfulness"] = float(score)

    return df

In [ ]:
def save_evaluation_file(rag_df: pd.DataFrame) -> None:
    output_df = rag_df[
        [
            "financebench_id",
            "question",
            "correctness",
            "faithfulness",
            "page_hit_at_1",
            "page_hit_at_3",
            "page_hit_at_5",
        ]
    ].copy()

    output_df.to_excel(TASK6_OUTPUT_XLSX, index=False)
    print(f"Saved Task 6 evaluation file to: {TASK6_OUTPUT_XLSX}")

In [ ]:
def compute_aggregate_scores(rag_df: pd.DataFrame) -> dict:
    return {
        "average_correctness": pd.to_numeric(rag_df["correctness"], errors="coerce").mean(),
        "average_faithfulness": pd.to_numeric(rag_df["faithfulness"], errors="coerce").mean(),
        "page_hit_at_1": rag_df["page_hit_at_1"].mean(),
        "page_hit_at_3": rag_df["page_hit_at_3"].mean(),
        "page_hit_at_5": rag_df["page_hit_at_5"].mean(),
    }

In [ ]:
# =======================
# Task 6 Execution Flow
# =======================

# Step 1: Load the full filtered FinanceBench dataset
financebench_df = load_clean_financebench_dataset()

# Step 2: Run RAG on all questions with k=5
print("building rag...")
rag_df = build_rag_df(financebench_df, k=5)
print("building rag end.")

# Initialize results_df with rag_df
results_df = rag_df.copy()

# Step 3: Add page-hit@k columns
print("adding page hit at k columns...")
results_df = add_page_hit_at_k_columns(results_df)
print("adding page hit at k columns end.")

# Step 5: Add faithfulness scores for the first 20 examples
print("adding faithfulness scores...")
results_df = add_faithfulness_scores(results_df, n=20)
print("adding faithfulness scores end.")

# Step 4: Add correctness scores
print("adding correctness scores...")
results_df = add_correctness_scores(results_df)
print("adding correctness scores end.")

# Step 6: Save the required xlsx file
print("saving evaluation file...")
save_evaluation_file(results_df)
print("saving evaluation file end.")

# Step 7: Compute and print aggregate metrics
print("computing aggregate metrics...")
summary = compute_aggregate_scores(results_df)
print("computing aggregate metrics end.")

print(summary)

## Task 6 results

The RAG pipeline achieved **average correctness = 0.375**, **average faithfulness = 0.760**, and **page-hit@1 / @3 / @5 = 0.19 / 0.31 / 0.39**.

This suggests that the system is **usually faithful to the retrieved context when answering**, but retrieval quality is still limited: even at `k=5`, the correct evidence page is retrieved only 39% of the time. The relatively low correctness score is therefore consistent with retrieval being the main bottleneck - the model often stays grounded in the retrieved text, but the retrieved text is not the right evidence often enough.

Overall, the gap between **faithfulness (high)** and **correctness (low)** indicates that the generation model is mostly behaving conservatively and using the provided context, but the retriever still misses the correct page too frequently.

## Task 7 – Motivation and planned experiments

The Task 6 baseline suggests that the main bottleneck is **retrieval rather than generation**.  
Faithfulness was relatively high, which means the model usually stayed grounded in the retrieved context, but correctness and page-hit@k were much lower, indicating that the right evidence page often was not retrieved in the first place.

Based on this, I chose experiments that primarily target retrieval quality, plus one lighter experiment on generation behavior.  
The goal is to test whether improving what reaches the model leads to better end-to-end correctness, while also checking whether any change unexpectedly hurts grounding or ranking quality.

### Planned experiments

1. **Increase top-k retrieval**
   - **Hypothesis:** Increasing the number of retrieved chunks should improve page-hit@k and may improve faithfulness because the correct evidence page is more likely to appear in the retrieved set. However, correctness may stay flat or decrease if extra irrelevant chunks distract the generator.

2. **Reduce chunk size**
   - **Hypothesis:** Using smaller chunks should improve retrieval precision and page-hit because the retrieved text may align more tightly with the evidence spans in the dataset. On the other hand, correctness or faithfulness could drop if the chunks become too narrow and lose useful surrounding context.

3. **Add a reranker**
   - **Hypothesis:** Adding a second-pass reranker should improve retrieval quality by pushing the most relevant evidence chunks closer to the top of the ranking. This is expected to improve correctness and page-hit@k, especially in cases where FAISS retrieves semantically related but wrong filings or pages.


In [ ]:
!pip -q install sentence-transformers

In [ ]:
TASK7_OUTPUT_XLSX = OUTPUT_DIR / "assignment2_improvement_cycles.xlsx"

TASK7_FAITHFULNESS_N = 20
TASK7_PAGE_HITS = [1, 3, 5]

BASELINE_K = 5
CHUNK_SIZES = [1000, 500, 1500, 2000, 300]

RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"

In [ ]:
from sentence_transformers import CrossEncoder

def load_reranker(model_name: str = RERANKER_MODEL_NAME):
    return CrossEncoder(model_name)

def rerank_documents(query: str, docs, reranker, top_k: int = 4):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc, _ in ranked[:top_k]]

In [ ]:
def answer_with_rag_v2(
    query: str,
    vectorstore,
    initial_k: int = 4,
    final_k: int | None = None,
) -> dict:
    retrieved_docs = vectorstore.similarity_search(query, k=initial_k)
    if final_k is not None:
        reranker = load_reranker()
        retrieved_docs = rerank_documents(query, retrieved_docs, reranker, top_k=final_k)

    messages = build_rag_messages(query, retrieved_docs)

    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        temperature=0
    )

    answer = response.choices[0].message.content.strip()

    retrieved_chunks = []
    for doc in retrieved_docs:
        retrieved_chunks.append({
            "doc_name": doc.metadata.get("doc_name"),
            "page_number": doc.metadata.get("page_number"),
            "content": doc.page_content,
        })

    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
    }

In [ ]:
from typing import Any, Callable

def evaluate_experiment(
    financebench_df: pd.DataFrame,
    answer_fn: Callable[[str], dict],
    faithfulness_n: int = TASK7_FAITHFULNESS_N,
) -> tuple[pd.DataFrame, dict]:
    rows = []

    for row in financebench_df.itertuples(index=False):
        result = answer_fn(row.question)

        expected_page_nums = extract_evidence_page_nums_from_evidence(row.evidence)

        rows.append({
            "financebench_id": row.financebench_id,
            "question": row.question,
            "ground_truth": row.answer,
            "expected_page_nums": expected_page_nums,
            "rag_answer": result["answer"],
            "retrieved_sources": result["retrieved_chunks"],
        })

    results_df = pd.DataFrame(rows)

    # page hit columns
    for k in TASK7_PAGE_HITS:
        results_df[f"page_hit_at_{k}"] = results_df.apply(
            lambda r: has_page_hit_at_k(r["expected_page_nums"], r["retrieved_sources"], k=k),
            axis=1
        )

    # correctness
    correctness_scores = []
    for row in results_df.itertuples(index=False):
        score = judge_correctness(
            question=row.question,
            ground_truth=row.ground_truth,
            rag_answer=row.rag_answer,
        )
        correctness_scores.append(score)

    results_df["correctness"] = correctness_scores

    # faithfulness
    results_df = add_faithfulness_scores(results_df, n=faithfulness_n)

    summary = {
        "correctness": pd.to_numeric(results_df["correctness"], errors="coerce").mean(),
        "faithfulness": pd.to_numeric(results_df["faithfulness"], errors="coerce").mean(),
        "page_hit_at_1": results_df["page_hit_at_1"].mean(),
        "page_hit_at_3": results_df["page_hit_at_3"].mean(),
        "page_hit_at_5": results_df["page_hit_at_5"].mean(),
    }

    return results_df, summary

In [ ]:
embedding_model = build_embedding_model()

## build faiss chunk indices
for chunk_size in CHUNK_SIZES:
    chunk_overlap = round(chunk_size * 0.175)
    index_file_path = INDEX_DIR / f"faiss_chunk{str(chunk_size)}"

    if not index_file_path.exists():
        text_splitter = build_text_splitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        chunk_docs = text_splitter.split_documents(page_docs)
        vectorstore = build_faiss_vectorstore(
            chunk_docs=chunk_docs,
            embedding_model=embedding_model,
            index_file_path=index_file_path,
        )
        save_faiss_vectorstore(vectorstore, index_file_path)

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS index /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk2000...
Saved FAISS index to: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk2000
Building FAISS index /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk300...
Saved FAISS index to: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk300


In [ ]:
baseline_vectorstore = load_faiss_vectorstore(FAISS_INDEX_PATH, embedding_model)
chunk500_vectorstore = load_faiss_vectorstore(INDEX_DIR / "faiss_chunk500", embedding_model)
chunk1500_vectorstore = load_faiss_vectorstore(INDEX_DIR / "faiss_chunk1500", embedding_model)
chunk300_vectorstore = load_faiss_vectorstore(INDEX_DIR / "faiss_chunk300", embedding_model)
chunk2000_vectorstore = load_faiss_vectorstore(INDEX_DIR / "faiss_chunk2000", embedding_model)

FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk500
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1500
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk300
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk2000


In [ ]:
def baseline_answer_fn(query: str) -> dict:
    return answer_with_rag_v2(
        query=query,
        vectorstore=baseline_vectorstore,
        initial_k=5,
    )


def exp_k8_answer_fn(query: str) -> dict:
    return answer_with_rag_v2(
        query=query,
        vectorstore=baseline_vectorstore,
        initial_k=8,
    )


def exp_chunk500_answer_fn(query: str) -> dict:
    return answer_with_rag_v2(
        query=query,
        vectorstore=chunk500_vectorstore,
        initial_k=5,
    )


def exp_reranker_answer_fn(query: str) -> dict:
    return answer_with_rag_v2(
        query=query,
        vectorstore=baseline_vectorstore,
        initial_k=20,
        final_k=4,
    )

In [ ]:
experiments = [
    {
        "experiment": "baseline",
        "change": "Task 6 baseline: chunk_size=1000, k=5, baseline prompt, no reranker",
        "hypothesis": "Baseline reference from Task 6.",
        "interpretation": "",
        "answer_fn": baseline_answer_fn,
    },
    {
        "experiment": "exp_k8",
        "change": "Increase generator retrieval k from 5 to 8",
        "hypothesis": "Increasing k should improve page-hit and may improve faithfulness because the correct page is more likely to be retrieved, but correctness may drop if extra chunks distract the generator.",
        "interpretation": "",
        "answer_fn": exp_k8_answer_fn,
    },
    {
        "experiment": "exp_chunk500",
        "change": "Reduce chunk size from 1000 to 500 and rebuild FAISS index",
        "hypothesis": "Smaller chunks should improve retrieval precision and page-hit by aligning more tightly to evidence spans, though they may lose some surrounding context.",
        "interpretation": "",
        "answer_fn": exp_chunk500_answer_fn,
    },
    {
        "experiment": "exp_reranker",
        "change": "Retrieve top-20 with FAISS and rerank to top-4 with BAAI/bge-reranker-base",
        "hypothesis": "A reranker should improve correctness and page-hit by pushing the most relevant evidence closer to the top of the ranking.",
        "interpretation": "",
        "answer_fn": exp_reranker_answer_fn,
    },
]

In [ ]:
financebench_df = load_clean_financebench_dataset()

task7_summary_rows = []
task7_full_results = {}

for exp in experiments:
    print(f"Running experiment: {exp['experiment']}")

    results_df, summary = evaluate_experiment(
        financebench_df=financebench_df,
        answer_fn=exp["answer_fn"],
        faithfulness_n=TASK7_FAITHFULNESS_N,
    )

    results_df.to_excel(CACHE_DIR / f"{exp['experiment']}_full_results.xlsx", index=False)

    task7_full_results[exp["experiment"]] = results_df

    task7_summary_rows.append({
        "experiment": exp["experiment"],
        "change": exp["change"],
        "correctness": summary["correctness"],
        "faithfulness": summary["faithfulness"],
        "page_hit_at_1": summary["page_hit_at_1"],
        "page_hit_at_3": summary["page_hit_at_3"],
        "page_hit_at_5": summary["page_hit_at_5"],
        "hypothesis": exp["hypothesis"],
        "interpretation": exp["interpretation"],
    })

task7_summary_df = pd.DataFrame(task7_summary_rows)

with pd.ExcelWriter(TASK7_OUTPUT_XLSX, engine="openpyxl") as writer:
    task7_summary_df.to_excel(writer, sheet_name="summary", index=False)

    for exp_name, df in task7_full_results.items():
        sheet_name = exp_name[:31]  # Excel sheet name limit
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved Task 7 results to: {TASK7_OUTPUT_XLSX}")



## Task 7 – Results and Interpretation

The baseline from Task 6 achieved correctness of **0.39**, faithfulness of **0.828**, and page-hit@1/@3/@5 of **0.19 / 0.31 / 0.39**.

### Experiment 1 – Increase k to 8
Increasing the number of retrieved chunks slightly improved correctness from **0.39** to **0.405**, but faithfulness dropped from **0.828** to **0.735**. Page-hit@k did not improve, suggesting that simply passing more context to the generator did not retrieve better evidence and may have added noise.

### Experiment 2 – Chunk size 500
Reducing chunk size from 1000 to 500 did not improve retrieval: page-hit@1/@3/@5 stayed at **0.19 / 0.31 / 0.39**. Correctness also slightly decreased to **0.38**, suggesting that smaller chunks did not solve the evidence-matching problem and may have lost some useful context.

### Experiment 3 – Reranker
The reranker performed worse than expected. Correctness dropped to **0.34**, and page-hit@1/@3/@5 decreased to **0.14 / 0.21 / 0.25**. This suggests that the reranker did not align well with the FinanceBench evidence-page objective, or that the initial top-20 candidates often did not contain enough correct evidence to rerank successfully.

## Wrap-up

Overall, the pipeline still fails mostly at the **retrieval stage**. The relatively high faithfulness scores show that the model often stays grounded in the retrieved context, but the low page-hit scores show that the retrieved context often does not contain the correct evidence page. If I had one more week, I would focus on hybrid retrieval, metadata-aware filtering by document/company/period, query rewriting, and testing rerankers more carefully against FinanceBench-style financial evidence.

# Bonus – Multi-scale Chunking

The hypothesis is that no single chunk size is optimal for all queries.  
To test this on FinanceBench, I compare retrieval performance across multiple FAISS indices built with different chunk sizes, while keeping the embedding model, splitter type, and overlap policy fixed.

The only variable changed between indices is `chunk_size`.
For each index, I retrieve top-5 chunks for every question and compute whether at least one retrieved chunk comes from the expected evidence page.

In [ ]:
BONUS_TOP_K = 5
BONUS_CHUNK_SIZES = [300, 500, 1000, 1500, 2000]

BONUS_OUTPUT_XLSX = OUTPUT_DIR / "assignment2_bonus_multiscale_chunking.xlsx"

In [ ]:
def create_hit_by_chunk_size_df(financebench_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    vs_map = {}

    ## preload vectorstores
    for chunk_size in BONUS_CHUNK_SIZES:
      vs_map[chunk_size] = load_faiss_vectorstore(
          INDEX_DIR / f"faiss_chunk{str(chunk_size)}",
          embedding_model,
      )

    for row in financebench_df.itertuples(index=False):
        expected_pages = extract_evidence_page_nums_from_evidence(row.evidence)

        result = {
            "financebench_id": row.financebench_id,
            "question": row.question,
        }

        for chunk_size in BONUS_CHUNK_SIZES:
            rag_query_result = answer_with_rag_v2(
                query=row.question,
                vectorstore=vs_map[chunk_size],
                initial_k=BONUS_TOP_K,
            )
            hit = has_page_hit_at_k(
                expected_pages=expected_pages,
                retrieved_sources=rag_query_result["retrieved_chunks"],
                k=BONUS_TOP_K,
              )
            result[f"hit_{chunk_size}"] = hit

        rows.append(result)

    return pd.DataFrame(rows)

In [ ]:
def create_hit_by_chunk_size_summary_df(results_df: pd.DataFrame) -> pd.DataFrame:
    summary = {}

    for chunk_size in BONUS_CHUNK_SIZES:
        summary[chunk_size] = results_df[f"hit_{chunk_size}"].mean()

    return pd.DataFrame([
        {"chunk_size": k, "page_hit_at_5": v}
        for k, v in summary.items()
    ])

In [ ]:
hit_cols = [f"hit_{c}" for c in BONUS_CHUNK_SIZES]

hit_by_chunk_size_df = create_hit_by_chunk_size_df(financebench_df)
hit_by_chunk_size_df["disagreement"] = hit_by_chunk_size_df[hit_cols].nunique(axis=1) > 1
hit_by_chunk_size_summary_df = create_hit_by_chunk_size_summary_df(hit_by_chunk_size_df)

disagreement_rate = hit_by_chunk_size_df["disagreement"].mean()

print("Disagreement rate:", disagreement_rate)

FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk300
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk500
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1000
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk1500
FAISS vector store loaded from: /content/drive/MyDrive/nebius academy/assignments/guy-peer_assignment2_rag/indices/faiss_chunk2000
Disagreement rate: 0.31


In [ ]:
hit_by_chunk_size_df.to_excel(OUTPUT_DIR / "bonus_per_question.xlsx", index=False)
hit_by_chunk_size_summary_df.to_excel(OUTPUT_DIR / "bonus_summary.xlsx", index=False)

## Bonus – Multi-scale Chunking Results

I tested five FAISS indices with different chunk sizes while keeping the embedding model, chunking method, and overlap policy fixed. The only variable changed was `chunk_size`.

| Chunk size | Page-hit@5 |
|---:|---:|
| 300 | 0.32 |
| 500 | 0.39 |
| 1000 | 0.39 |
| 1500 | 0.39 |
| 2000 | 0.33 |

The best-performing chunk sizes were **500, 1000, and 1500**, all tied at **page-hit@5 = 0.39**. Very small chunks (`300`) and very large chunks (`2000`) performed worse.

The disagreement rate across questions was **0.31**, meaning that for about **31% of questions**, changing the chunk size changed whether retrieval hit the correct evidence page.

Overall, the results partially support the article’s claim that no single chunk size is optimal for all queries. There was no unique dominant winner, but there was a clear preference for medium-sized chunks in this FinanceBench setup. This suggests that chunk size is query-dependent, but the dataset seems to favor a middle range rather than extreme small or large chunks.